# Lecture 5 (optional): Learning a potential energy curve with neural networks and Gaussian processes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Edinburgh-Chemistry-Teaching/AdvancedPhysicalChemistry/blob/main/L5_neural_networks/L5_neural_networks_LJ.ipynb)

<a rel="license" href="https://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons Licence" style="border-width:0" src="https://licensebuttons.net/l/by-sa/4.0/88x31.png" title='This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.' align="right"/></a>

Author: Dr Antonia Mey
Email: antonia.mey@ed.ac.uk

**Time needed:** about 30 minutes. This notebook is **optional** practice for Lecture 5 (neural networks and machine-learned potentials).

Solutions are in the green **Click here to see solution** drop-downs below each task. Try the task yourself first, then open the drop-down to check.

### Learning objectives
By the end of this notebook, you should be able to
* Fit a small neural network (`MLPRegressor`) to energies and compare it with the true potential.
* Read training and validation loss curves, spot overfitting and say when to stop training.
* Fit a Gaussian process and use its uncertainty band, or the spread of several networks, to see where a model has no data.
* Explain why no model can be trusted far outside its training data.

### Table of Contents

0. [The Lennard-Jones potential as a test case](#0-lj)
1. [A neural network learns U(r)](#1-nn)
2. [Loss curves and overfitting](#2-loss)
3. [Where is the model guessing? Gaussian process vs an ensemble of networks](#3-gp)
4. [Key points](#4-key-points)

### How to open this notebook

* **Noteable (recommended):** open it from the Lecture 5 page on Learn. Everything you need is already installed.
* **Google Colab (backup):** click the *Open in Colab* button at the top. You need a Google account. Save a copy to your Drive (*File → Save a copy in Drive*) or your answers will be lost.
* **Your own computer:** you need `numpy`, `matplotlib` and `scikit-learn` (see `requirements.txt` in the repository).

**<span style="color:black">Jupyter Cheat Sheet</span>**
- To run the currently highlighted cell and move focus to the next cell, hold <kbd>&#x21E7; Shift</kbd> and press <kbd>&#x23ce; Enter</kbd>;
- To get help for a specific function, place the cursor within the function's brackets, hold <kbd>&#x21E7; Shift</kbd>, and press <kbd>&#x21E5; Tab</kbd>;
- Cells marked `# FIXME` are for you to complete.

# Import libraries

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

warnings.filterwarnings("ignore")   # MLPRegressor warns when it stops before full convergence
BLUE, ORANGE, LAV = '#2a78d6', '#eb6834', '#a897d1'

# 0. The Lennard-Jones potential as a test case <a id="0-lj"></a>

In a real machine-learned potential the training energies come from quantum chemistry (e.g. DFT), and each calculation is expensive. Here we use the Lennard-Jones (LJ) pair potential instead,
$$U(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right].$$
We work in reduced units: distances in units of $\sigma$ and energies in units of $\varepsilon$ (so $\sigma = \varepsilon = 1$), which gives
$$U(r) = 4\left[\left(\frac{1}{r}\right)^{12} - \left(\frac{1}{r}\right)^{6}\right].$$

The advantage: we know the true answer everywhere, so we can see exactly where a model is right and where it is guessing.

As in the lecture, our "reference calculations" are 15 values of $U(r)$: close together between 1.0 and 1.5 $\sigma$, then every 0.15 $\sigma$ between 2.1 and 3.0 $\sigma$. There is a **gap** between 1.5 and 2.1 $\sigma$ and **no data** below 1.0 $\sigma$ or above 3.0 $\sigma$.

In [ ]:
def U(r):
    """Lennard-Jones pair energy in reduced units."""
    return 4 * (r ** -12 - r ** -6)


r_train = np.r_[1.0, 1.05, 1.1, 1.15, 1.2, 1.3, 1.4, 1.5, np.arange(2.1, 3.01, 0.15)]
E_train = U(r_train)
r_plot = np.linspace(0.9, 3.5, 700)          # where we will ask the models for predictions


def shade_no_data(ax):
    """Grey bands where there is no training data."""
    for lo, hi in [(0.9, 1.0), (1.5, 2.1), (3.0, 3.5)]:
        ax.axvspan(lo, hi, color="0.92", lw=0, zorder=0)


fig, ax = plt.subplots(figsize=(7, 4))
shade_no_data(ax)
ax.plot(r_plot, U(r_plot), "--", color="grey", label="true U(r)")
ax.plot(r_train, E_train, "o", color=BLUE, label="training data")
ax.set_ylim(-1.3, 3); ax.set_xlabel("r / σ"); ax.set_ylabel("U / ε"); ax.legend()
print(len(r_train), "training points")

# 1. A neural network learns U(r) <a id="1-nn"></a>

`MLPRegressor` is scikit-learn's fully connected neural network (multilayer perceptron). It is used exactly like the models in Lectures 1–4: `fit(X, y)`, then `predict(X_new)`.

* `hidden_layer_sizes=(20, 20)`: two hidden layers with 20 neurons each.
* `activation="tanh"`: the non-linear activation function in every hidden neuron.
* `solver="lbfgs"`: the minimiser used to bring the loss (mean squared error) down. For a small data set it works better than plain gradient descent.
* `random_state`: the random starting weights. Different starts can end in different minima of the loss.

scikit-learn expects a 2D input array (samples × features), so the single feature $r$ becomes a column with `r[:, None]`.

In [ ]:
net = MLPRegressor(hidden_layer_sizes=(20, 20), activation="tanh", solver="lbfgs",
                   alpha=1e-6, max_iter=20000, tol=1e-10, random_state=0)
net.fit(r_train[:, None], E_train)
E_nn = net.predict(r_plot[:, None])

fig, ax = plt.subplots(figsize=(7, 4))
shade_no_data(ax)
ax.plot(r_plot, U(r_plot), "--", color="grey", label="true U(r)")
ax.plot(r_plot, E_nn, color=ORANGE, label="neural network")
ax.plot(r_train, E_train, "o", color=BLUE, label="training data")
ax.set_ylim(-1.3, 3); ax.set_xlabel("r / σ"); ax.set_ylabel("U / ε"); ax.legend()

n_par = sum(w.size for w in net.coefs_) + sum(b.size for b in net.intercepts_)
print("number of weights and biases:", n_par)
print("largest error at the training points:", np.abs(net.predict(r_train[:, None]) - E_train).max().round(4))

## Tasks 1 <a id="tasks-1"></a>
1. Check the parameter count printed above by hand: 1 input, two hidden layers of 20 neurons, 1 output, with one bias per neuron.
2. Refit the network with `random_state` = 1, 2 and 3 and plot all of them on one graph. In which regions of $r$ do the fits agree, and where do they differ?
3. Try the smallest possible network, one hidden neuron (`hidden_layer_sizes=(1,)`), and then two (`(2,)`, several seeds). Is one neuron enough to describe the potential well?

In [ ]:
# 1. Parameter count by hand
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 1.1 </summary>

```python
# 1. Parameter count by hand
# input -> hidden 1: 1*20 weights + 20 biases; hidden 1 -> hidden 2: 20*20 + 20; hidden 2 -> output: 20*1 + 1
print(1 * 20 + 20 + 20 * 20 + 20 + 20 * 1 + 1)   # 481, the same as n_par
```

</details>

In [ ]:
# 2. Different random starting weights
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 1.2 </summary>

```python
# 2. Different random starting weights
fig, ax = plt.subplots(figsize=(7, 4))
shade_no_data(ax)
ax.plot(r_plot, U(r_plot), "--", color="grey", label="true U(r)")
for seed in range(4):
    m = MLPRegressor(hidden_layer_sizes=(20, 20), activation="tanh", solver="lbfgs",
                     alpha=1e-6, max_iter=20000, tol=1e-10, random_state=seed).fit(r_train[:, None], E_train)
    ax.plot(r_plot, m.predict(r_plot[:, None]), color=ORANGE, lw=1, label=f"seed {seed}")
ax.plot(r_train, E_train, "o", color=BLUE)
ax.set_ylim(-1.3, 3); ax.set_xlabel("r / σ"); ax.set_ylabel("U / ε"); ax.legend()
plt.show()
```

</details>

In [ ]:
# 3. A very small network
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 1.3 </summary>

```python
# 3. A very small network
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r_plot, U(r_plot), "--", color="grey", label="true U(r)")
for hidden, colour in [((1,), LAV), ((2,), ORANGE)]:
    for seed in range(3):
        small = MLPRegressor(hidden_layer_sizes=hidden, activation="tanh", solver="lbfgs", alpha=1e-6,
                             max_iter=20000, tol=1e-10, random_state=seed).fit(r_train[:, None], E_train)
        err = np.abs(small.predict(r_train[:, None]) - E_train).max()
        print(f"{hidden[0]} hidden neuron(s), seed {seed}: largest error at the training points {err:.3f}")
        ax.plot(r_plot, small.predict(r_plot[:, None]), color=colour, lw=1)
ax.plot(r_train, E_train, "o", color=BLUE)
ax.set_ylim(-1.3, 3); ax.set_xlabel("r / σ"); ax.set_ylabel("U / ε")
plt.show()
```

</details>

*Your answers to questions 2 and 3 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to questions 2 and 3 (Tasks 1) </summary>

All seeds go through the training points and agree closely between 1.0 and 3.0 $\sigma$, including the gap, where the curve is smooth. They differ most below 1.0 $\sigma$, on the repulsive wall, where there is no data. Every network makes its own guess, and all of them make the wall too soft. One hidden neuron is a single tanh step and it can only go down and level off, so it misses the minimum completely (largest error about 0.7 $\epsilon$, underfitting as in Lecture 1). Two neurons can make one bump, which is just what the well needs (error about 0.006 $\epsilon$), but only from some starting weights. Other seeds get stuck at the one-step solution, a local minimum of the loss. The number of neurons controls how flexible the model is, like the degree of a polynomial.

</details>

# 2. Loss curves and overfitting <a id="2-loss"></a>

A real potential depends on more than one distance. As in the lecture, we now learn the energy of three LJ atoms (a **trimer**) in 2D. The input is the three **sorted** pair distances (an invariant descriptor, Lecture 3) and the target is the total energy $E = U(r_{12}) + U(r_{13}) + U(r_{23})$. We add a little random noise to the energies to mimic the numerical noise of a reference calculation.

We train the network **one epoch at a time** with `partial_fit` (one pass through all training data in mini-batches, with the Adam optimiser) and record the mean squared error (MSE) on the training set and on a separate **validation set** after every epoch.

In [ ]:
def trimers(n, rng):
    """n random 2D LJ trimers: sorted pair distances (n x 3) and total energies (n)."""
    X, E = [], []
    while len(X) < n:
        p = rng.uniform(0, 2.2, (3, 2))                 # three atoms in a 2.2 x 2.2 sigma box
        d = np.sort([np.linalg.norm(p[0] - p[1]), np.linalg.norm(p[0] - p[2]), np.linalg.norm(p[1] - p[2])])
        if d[0] >= 0.95:                                # skip atoms that overlap
            X.append(d); E.append(U(d).sum())
    return np.array(X), np.array(E)


def loss_curves(n_train, hidden, noise, epochs, seed=0):
    """Train an MLP epoch by epoch; return training and validation MSE after every epoch."""
    rng = np.random.default_rng(seed)
    X_tr, E_tr = trimers(n_train, rng)
    X_va, E_va = trimers(500, rng)
    E_tr = E_tr + rng.normal(0, noise, n_train)
    E_va = E_va + rng.normal(0, noise, 500)
    scaler = StandardScaler().fit(X_tr)                 # scale features using the training set only
    X_tr, X_va = scaler.transform(X_tr), scaler.transform(X_va)
    net = MLPRegressor(hidden_layer_sizes=hidden, learning_rate_init=1e-3,
                       batch_size=min(32, n_train), random_state=seed)
    train, val = [], []
    for epoch in range(epochs):
        net.partial_fit(X_tr, E_tr)                     # one epoch
        train.append(np.mean((net.predict(X_tr) - E_tr) ** 2))
        val.append(np.mean((net.predict(X_va) - E_va) ** 2))
    return np.array(train), np.array(val)


def plot_loss(train, val, title):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(train, color=BLUE, label="training")
    ax.plot(val, color=LAV, label="validation")
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE / ε²")
    ax.set_title(title); ax.legend()
    plt.show()


# Model A: many training trimers, moderate network (takes about 20 s)
train_A, val_A = loss_curves(n_train=2000, hidden=(32, 32), noise=0.05, epochs=150)
plot_loss(train_A, val_A, "A: 2000 training trimers, 2 x 32 neurons")
print(f"final MSE  training {train_A[-1]:.4f}   validation {val_A[-1]:.4f}")

## Tasks 2 <a id="tasks-2"></a>
1. Train model B with only 30 training trimers, a large network `hidden=(200, 200)`, `noise=0.15` and 600 epochs, and plot its loss curves.
2. Compare A and B. Which one is overfitting, and how can you tell from the curves alone?
3. Find the epoch with the lowest validation MSE for model B (`np.argmin`). Why would you stop training there (early stopping), and why must the validation set not be used for training?

In [ ]:
# 1. Model B: few data, big network
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 2.1 </summary>

```python
# 1. Model B: few data, big network
train_B, val_B = loss_curves(n_train=30, hidden=(200, 200), noise=0.15, epochs=600)
plot_loss(train_B, val_B, "B: 30 training trimers, 2 x 200 neurons")
print(f"final MSE  training {train_B[-1]:.5f}   validation {val_B[-1]:.3f}")
```

</details>

In [ ]:
# 3. Early stopping
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 2.3 </summary>

```python
# 3. Early stopping
best = np.argmin(val_B)
print(f"lowest validation MSE {val_B[best]:.3f} at epoch {best + 1}; "
      f"at the end: {val_B[-1]:.3f}")
```

</details>

*Your answers to questions 2 and 3 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to questions 2 and 3 (Tasks 2) </summary>

In model A the two curves fall together and end close to each other, near the noise level (0.05² = 0.0025 ε²): the model has learned the energy function. Model B is overfitting: its training MSE keeps falling towards zero (it memorises 30 trimers, noise included) while the validation MSE flattens out around 0.2 ε², ten times the noise level (0.15² = 0.0225 ε²) and many orders of magnitude above the training MSE. The signature is the growing gap between the two curves, not the value of the training loss. We stop at the validation minimum because after that, further training only fits the training noise. The validation set must stay separate: if it were used to train the weights, it would no longer tell us how the model does on new configurations (the same reason as the test set in Lectures 1 and 4). The real cure for B is more data or a smaller network.

</details>

# 3. Where is the model guessing? Gaussian process vs an ensemble of networks <a id="3-gp"></a>

A **Gaussian process** (GP) predicts the energy at a new $r$ as a similarity-weighted average of the training energies. The similarity is the kernel given by:

$$k(r, r') = 0.5^2 \exp\left[-\frac{(r - r')^2}{2 \times 0.15^2}\right],$$

meaning that points closer than about 0.15 $\sigma$ count as similar. The GP also returns a standard deviation: small near the data, large far from it.

We fix the kernel (`optimizer=None`) so that you get the same result as the lecture figure. In practice the kernel parameters are also fitted to the data.

In [ ]:
kernel = ConstantKernel(0.5 ** 2, "fixed") * RBF(length_scale=0.15, length_scale_bounds="fixed")
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, optimizer=None)
gp.fit(r_train[:, None], E_train)
mu, sd = gp.predict(r_plot[:, None], return_std=True)

fig, ax = plt.subplots(figsize=(7, 4))
shade_no_data(ax)
ax.plot(r_plot, U(r_plot), "--", color="grey", label="true U(r)")
ax.fill_between(r_plot, mu - 2 * sd, mu + 2 * sd, color=LAV, alpha=0.3, lw=0, label="GP ± 2 sd")
ax.plot(r_plot, mu, color=LAV, lw=2, label="GP mean")
ax.plot(r_train, E_train, "o", color=BLUE, label="training data")
ax.set_ylim(-1.3, 3); ax.set_xlabel("r / σ"); ax.set_ylabel("U / ε"); ax.legend()
plt.show()


def report(name, lo, hi):
    """Largest GP error and largest GP uncertainty (2 sd) for lo < r < hi."""
    w = (r_plot > lo) & (r_plot < hi)
    print(f"{name:12s} GP max error {np.abs(mu - U(r_plot))[w].max():.2f}   GP max 2 sd {2 * sd[w].max():.2f}")


report("gap", 1.5, 2.1)
report("wall < 1.0", 0.9, 1.0)

## Tasks 3 <a id="tasks-3"></a>
1. Train 8 networks as in Section 1 (seeds 0–7) and store their predictions on `r_plot` in an array `P` of shape (8, 700). Plot them together with the GP mean and band.
2. An **ensemble** of networks gives a cheap uncertainty: where they disagree, at least some of them must be wrong. Compute the spread (`np.ptp(P, axis=0)`) and the largest error of the networks, in the gap (1.5–2.1 $\sigma$) and on the wall (below 1.0 $\sigma$). Compare with the GP numbers above.
3. Which model would you trust in the gap, and which below 1.0 $\sigma$? What would you do before running MD that reaches these distances?

In [ ]:
# 1. Ensemble of 8 networks vs the GP
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 3.1 </summary>

```python
# 1. Ensemble of 8 networks vs the GP
P = np.array([MLPRegressor(hidden_layer_sizes=(20, 20), activation="tanh", solver="lbfgs", alpha=1e-6,
                           max_iter=20000, tol=1e-10, random_state=seed)
              .fit(r_train[:, None], E_train).predict(r_plot[:, None]) for seed in range(8)])

fig, ax = plt.subplots(figsize=(7, 4))
shade_no_data(ax)
ax.plot(r_plot, U(r_plot), "--", color="grey", label="true U(r)")
ax.fill_between(r_plot, mu - 2 * sd, mu + 2 * sd, color=LAV, alpha=0.3, lw=0, label="GP ± 2 sd")
ax.plot(r_plot, mu, color=LAV, lw=2, label="GP mean")
for p in P:
    ax.plot(r_plot, p, color=ORANGE, lw=1)
ax.plot([], [], color=ORANGE, label="8 networks")
ax.plot(r_train, E_train, "o", color=BLUE)
ax.set_ylim(-1.3, 3); ax.set_xlabel("r / σ"); ax.set_ylabel("U / ε"); ax.legend()
plt.show()
```

</details>

In [ ]:
# 2. Spread and error of the ensemble
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 3.2 </summary>

```python
# 2. Spread and error of the ensemble
for name, lo, hi in [("gap", 1.5, 2.1), ("wall < 1.0", 0.9, 1.0)]:
    w = (r_plot > lo) & (r_plot < hi)
    print(f"{name:12s} NN max error {np.abs(P - U(r_plot))[:, w].max():.2f}   "
          f"NN spread {np.ptp(P, axis=0)[w].max():.2f}")
```

</details>

*Your answers to question 3 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to question 3 (Tasks 3) </summary>

In the gap the networks happen to agree with each other and with the true curve (error about 0.02 ε), because the curve is smooth there. The GP mean is worse (error about 0.2 ε), but the GP honestly says so: its 2 sd band grows to almost 1 ε. On the repulsive wall below 1.0 σ, both models are wrong. The networks are all too soft, but they also disagree with each other, which is a warning. The GP is confidently wrong: its band stays narrow (2 sd about 0.1 ε) while its error is about 3 ε. Uncertainty estimates help, but none of them is a guarantee outside the training data. Before running MD that reaches short distances (high temperature, collisions), add reference calculations there; choosing new points where the models are most uncertain is called *active learning*.

</details>

# 4. Key points <a id="4-key-points"></a>

* A neural network is fitted like any model in this course: choose a loss (MSE), minimise it, check on data it has not seen. The number of neurons sets how flexible it is.
* **Loss curves:** training and validation MSE falling together = learning; training falling while validation stalls or rises = overfitting. Stop at the validation minimum (early stopping), and get more data if you can.
* Networks started from different random weights fit the training data equally well but can disagree away from it. The **spread of an ensemble** is a cheap warning sign.
* A **Gaussian process** gives an uncertainty with every prediction: narrow near the data, wide in gaps. It is not a guarantee far from the data.
* For machine-learned potentials: the model is only as good as the configurations it was trained on, so sample the regions your simulation will visit.

**Questions to think about:**
1. Your ML potential for liquid water was trained on 300 K MD frames. Why might it fail for ice, or for water at 600 K?
2. Why would you rather predict the *energy* and take forces as $-\nabla E$, than train a second network for the forces?